# Customer-Support Agent Conversation Using DeepSeek-R1 Model

This notebook demonstrates how to generate a conversation between a customer and a support agent using the `deepseek-r1` model. The conversation focuses on analyzing agricultural startup spending, generating actionable insights, and summarizing the conversation.

## Features:
1. Analyze agricultural spending patterns.
2. Generate actionable insights using the `deepseek-r1` model.
3. Summarize the conversation and extract meaningful insights.

---

# Step 1: Install Required Dependencies.

In [1]:
# Install lshw module so that Ollama will be able to see hardware details
!sudo apt-get install -y lshw

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  pci.ids usb.ids
The following NEW packages will be installed:
  lshw pci.ids usb.ids
0 upgraded, 3 newly installed, 0 to remove and 29 not upgraded.
Need to get 790 kB of archives.
After this operation, 2,988 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2build1 [321 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 pci.ids all 0.0~2022.01.22-1 [251 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Fetched 790 kB in 1s (592 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf

In [2]:
# Run ollama serve in the background
import subprocess
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(f"Ollama serve process ID: {process.pid}")

# Pull deepseek-r1 model
!ollama pull deepseek-r1

Ollama serve process ID: 732
Error: could not connect to ollama app, is it running?


## Step 2: Define Functions for Spending Analysis, Insights Generation, and Conversation Summary

We'll define three key functions:
1. `analyze_agri_spending`: Analyzes agricultural spending patterns from a CSV file.
2. `get_agri_insights`: Uses the `deepseek-r1` model to generate actionable insights based on the analysis.
3. `summarize_conversation`: Generates a concise summary of the conversation and extracts meaningful insights.

In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json

def analyze_agri_spending(csv_path):
    """
    Analyze agricultural startup spending patterns.
    Dataset 'startup_funding.csv' path from kaggle =>
    https://www.kaggle.com/code/kabure/indian-startups-python-explorations/input
    df_startups = pd.read_csv("../input/startup_funding.csv",index_col=0)
    """

    # Read CSV file
    df = pd.read_csv(csv_path)

    analysis = []

    # Get numerical columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # 1. Overall Spending Categories
    analysis.append("=== MAJOR SPENDING AREAS ===")
    total_spending = df[numeric_cols].sum().sort_values(ascending=False)

    # Format as percentage of total
    total_sum = total_spending.sum()
    for category, amount in total_spending.items():
        percentage = (amount / total_sum) * 100
        analysis.append(f"{category}:")
        analysis.append(f"  Total: ${amount:,.2f}")
        analysis.append(f"  Percentage of Budget: {percentage:.1f}%")

    # 2. Monthly Trends
    analysis.append("\n=== MONTHLY SPENDING TRENDS ===")
    monthly_totals = df[numeric_cols].mean()
    analysis.append("Average Monthly Spending:")
    for category, amount in monthly_totals.items():
        analysis.append(f"{category}: ${amount:,.2f}/month")

    # 3. Key Metrics
    analysis.append("\n=== KEY SPENDING METRICS ===")
    for col in numeric_cols:
        analysis.append(f"\n{col}:")
        analysis.append(f"  Highest: ${df[col].max():,.2f}")
        analysis.append(f"  Lowest: ${df[col].min():,.2f}")
        analysis.append(f"  Average: ${df[col].mean():,.2f}")

        # Calculate month-over-month change
        if len(df) > 1:
            last_month = df[col].iloc[-1]
            prev_month = df[col].iloc[-2]
            if prev_month != 0:
                change = ((last_month - prev_month) / prev_month) * 100
                analysis.append(f"  Recent Change: {change:+.1f}%")

    # 4. Cost Categories Analysis
    analysis.append("\n=== COST BREAKDOWN ===")

    # Identify operational vs capital expenses
    operational_keywords = ['seed', 'fertilizer', 'labor', 'water', 'maintenance', 'supplies']
    capital_keywords = ['equipment', 'machinery', 'land', 'infrastructure', 'vehicle']

    op_expenses = 0
    cap_expenses = 0

    for col in numeric_cols:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in operational_keywords):
            op_expenses += df[col].sum()
        elif any(keyword in col_lower for keyword in capital_keywords):
            cap_expenses += df[col].sum()

    total_expenses = op_expenses + cap_expenses
    if total_expenses > 0:
        analysis.append(f"Operational Expenses: ${op_expenses:,.2f} ({(op_expenses/total_expenses)*100:.1f}%)")
        analysis.append(f"Capital Expenses: ${cap_expenses:,.2f} ({(cap_expenses/total_expenses)*100:.1f}%)")

    return "\n".join(analysis)

def get_agri_insights(analysis_text):
    """
    Get agricultural-specific insights using the deepseek-r1 model.
    """
    url = "http://localhost:11434/api/chat"
    prompt = f"""
    As an agricultural business expert, analyze this startup's spending:
    {analysis_text}

    Please provide:
    1. Main spending patterns and their alignment with agricultural cycles
    2. Areas where costs could be optimized
    3. Suggestions for resource allocation
    4. Recommendations for sustainable growth

    Focus on practical agricultural business insights.
    """

    payload = {
        "model": "deepseek-r1",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    try:
        response = requests.post(url, json=payload, stream=True)
        full_response = ""

        for line in response.iter_lines():
            if line:
                json_response = json.loads(line.decode('utf-8'))
                if 'message' in json_response:
                    content = json_response['message']['content']
                    full_response += content

        return full_response
    except Exception as e:
        return "Error getting AI insights. Proceeding with numerical analysis only."

def summarize_conversation(analysis_text, insights_text):
    """
    Generate a concise summary of the conversation and extract meaningful insights.
    """
    url = "http://localhost:11434/api/chat"
    prompt = f"""
    Summarize the following conversation between a customer and a support agent:

    ANALYSIS:
    {analysis_text}

    INSIGHTS:
    {insights_text}

    Extract key points and actionable insights into bullet points.
    """

    payload = {
        "model": "deepseek-r1",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    try:
        response = requests.post(url, json=payload, stream=True)
        full_response = ""

        for line in response.iter_lines():
            if line:
                json_response = json.loads(line.decode('utf-8'))
                if 'message' in json_response:
                    content = json_response['message']['content']
                    full_response += content

        return full_response
    except Exception as e:
        return "Error generating conversation summary."

## Step 3: Simulate a Conversation Between Customer and Support Agent

Let’s simulate the conversation by analyzing a sample dataset, generating insights, and summarizing the conversation.

In [5]:
# Upload your dataset (CSV file) to the Colab environment
from google.colab import files
uploaded = files.upload()

# Replace 'your_file.csv' with the name of your uploaded file
csv_path = list(uploaded.keys())[0]

# Analyze agricultural startup spending
spending_analysis = analyze_agri_spending(csv_path)
print("\n=== Agricultural Spending Analysis ===")
print(spending_analysis)

# Generate insights using deepseek-r1
insights = get_agri_insights(spending_analysis)
print("\n=== Expert Insights ===")
print(insights)

# Generate a conversation summary
summary = summarize_conversation(spending_analysis, insights)
print("\n=== Conversation Summary ===")
print(summary)

Saving startup_funding.csv to startup_funding.csv

=== Agricultural Spending Analysis ===
=== MAJOR SPENDING AREAS ===
SNo:
  Total: $2,812,006.00
  Percentage of Budget: 100.0%

=== MONTHLY SPENDING TRENDS ===
Average Monthly Spending:
SNo: $1,185.50/month

=== KEY SPENDING METRICS ===

SNo:
  Highest: $2,371.00
  Lowest: $0.00
  Average: $1,185.50
  Recent Change: +0.0%

=== COST BREAKDOWN ===

=== Expert Insights ===


=== Conversation Summary ===



## Notes:
1. Ensure the `deepseek-r1` model is running in the background before executing the notebook.
2. Replace the sample dataset with your own CSV file containing agricultural spending data.
3. The notebook assumes the dataset has numerical columns representing different spending categories.